### Setup
Autoreload and imports

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
# Imports
import sys, os
from pathlib import Path
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, SaliencyMapMethod, CarliniL2Method
from art.defences.trainer import AdversarialTrainer, AdversarialTrainerTRADESPyTorch
import art.attacks.evasion.projected_gradient_descent.projected_gradient_descent_pytorch as _pgd_pt
_pgd_pt.compute_success = lambda *a, **kw: 0.0

sys.path.append(str(Path.cwd().parents[1]))

from utils.functions import get_windowed_data
from utils.notebook import get_model_classifier, clean_data_test, adv_test, FilenameLoader, get_filename_from_path, freeze_attack_cols, freeze_benign_and_cols

### Define Inputs and Load Data

In [15]:
from dataclasses import dataclass

@dataclass
class AdvParams:
    end_index : int

@dataclass
class FgsmParams(AdvParams):
    eps : float


class TestLoader:
    def __init__ (self, checkpoint_file, data_file, name):
        self.checkpoint_file = checkpoint_file
        self.data_file = data_file
        self.name = name
        print(f"> Initialized with {self.name}")
    
    def load_data(self):
        (self.x_train, self.y_train), (self.x_test, self.y_test), fed_dataset, self.scaler = get_windowed_data(self.data_file, 
                                                                      normalize=True, 
                                                                      train_perc=80)
        self.x_train_np = self.x_train.numpy()
        self.y_train_np = self.y_train.numpy()
        print(f"> Loaded Data for {self.data_file}")

    def load_model(self):
        self.model, self.classifier = get_model_classifier(self.checkpoint_file)
        print(f"> Loaded Model for {self.checkpoint_file}")

    def clean_test(self, name : str, save_dir : str, save_results : bool = True):
        print("[!!] Running on *current* loaded model! To run on a different model, re-initialize TestLoader")
        os.makedirs(save_dir, exist_ok=True)
        clean_out = clean_data_test(
            self.model, self.classifier, self.x_test, self.y_test, 
            checkpoint_file=self.checkpoint_file, data_file=self.data_file,
            save_path=save_dir,
            filename=f"{name}.json",
            save_results=save_results
        )
        return clean_out 

    def fgsm_adv_test(self, save_dir : str, params : FgsmParams):
        print("[!!] Running on *current* loaded model! To run on a different model, re-initialize TestLoader")
        os.makedirs(save_dir, exist_ok=True)
        adv_out = adv_test(
            self.classifier, self.x_test, self.y_test, 
            checkpoint_file=self.checkpoint_file, data_file=self.data_file,
            end_index=params.end_index,
            path=save_dir+"/fgsm",
            filename=f"fgsm_adv_eps_{params.eps}.json",
            Attack=FastGradientMethod,
            eps=params.eps,
        )
        return adv_out


In [5]:
## Load checkpoint and data file paths
_, data_name, _ = FilenameLoader.rand_pos()

checkpoint_file= f"../../saved_models/adv_trained/RandPos-PGD-Evasion-30-30-200.ckpt"
data_file = f"../../data/{data_name}"

In [17]:
loader = TestLoader(checkpoint_file=checkpoint_file, 
           data_file=data_file, 
           name="RandPos-PGD-Evasion-30-30-200")
loader.load_data()
loader.load_model()

> Initialized with RandPos-PGD-Evasion-30-30-200


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/anaconda3/envs/reu/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


> Loaded Data for ../../data/RandomPos_0709.csv
Checkpoint path exists!
> Loaded Model for ../../saved_models/adv_trained/RandPos-PGD-Evasion-30-30-200.ckpt


### Adversarial training

#### Clean Testing

In [13]:
print("> Before Adv Test")

adv_save_dir = f"../fed/data-test/test"

for i in range(1, 31):
    eps = float(i/100)

    adv_out = loader.fgsm_adv_test(
        save_dir=adv_save_dir,
        params=FgsmParams(
            end_index=len(loader.y_test.numpy()),
            eps=eps
        )
    )
    

> Before Adv Test
=== Attack: FastGradientMethod, kwargs: {'eps': 0.01} ===
Accuracy:           0.9849
Precision:          0.9965
Recall:             0.9526
F1:                 0.9741
ASR (FNR):          0.0474
False Positive Rate:0.0014
TP=353129, TN=875800, FP=1240, FN=17571
Time elapsed:       14.66s
Saved metrics to ../fed/data-test/test/fgsm/fgsm_adv_eps_0.01.json
=== Attack: FastGradientMethod, kwargs: {'eps': 0.02} ===
Accuracy:           0.9823
Precision:          0.9948
Recall:             0.9453
F1:                 0.9694
ASR (FNR):          0.0547
False Positive Rate:0.0021
TP=350417, TN=875217, FP=1823, FN=20283
Time elapsed:       15.01s
Saved metrics to ../fed/data-test/test/fgsm/fgsm_adv_eps_0.02.json
=== Attack: FastGradientMethod, kwargs: {'eps': 0.03} ===


KeyboardInterrupt: 

In [16]:
# Get clean baseline 
print("> Before Clean Test")

before_clean_out = loader.clean_test(name = "clean.json",
                                     save_dir=adv_save_dir)

> Before Clean Test
torch.Size([124774, 10, 2])
0.9976138469777732
0.959778796870785
Model got 1231979/1247740 right.
Accuracy: 0.9873683619984933, Precision: 0.9976138469777732, Recall: 0.959778796870785, F1 Score: 0.9783306592093668
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
Saved to ../fed/data-test/test/clean.json_clean_advtrained_RandPos-PGD-Evasion-30-30-200.json


#### Adv Training

In [ ]:
## Simple training - pgd at 0.05
from art.defences.trainer import AdversarialTrainer
import torch 

# Training vars
adv_train_eps = 0.00
adv_train_epochs = 20
adv_train_ratio = 1.0

# Init vars
adv_save_dir = f"../fed/data-test/randpos2_pgd-{adv_train_eps}_epochs-{adv_train_epochs}_trained-30-30-200_ratio-{adv_train_ratio}"
os.makedirs(adv_save_dir, exist_ok=True)

# Get clean baseline 
print("> Before Clean Test")
before_clean_out = clean_data_test(
    model, classifier, x_test, y_test, 
    checkpoint_file=checkpoint_file, data_file=data_file,
    save_path=adv_save_dir,
    filename=f"before_clean_advtrained_{name}.json",
    save_results=True
)

# Check condition 1
print ("> Before Adv Test")
before_adv_out = adv_test(
    classifier, x_test, y_test, 
    checkpoint_file=checkpoint_file, data_file=data_file,
    end_index=len(y_test.numpy()),
    path=adv_save_dir,
    filename=f"before_adv_eps_{adv_train_eps}_advtrained_{name}.json",
    Attack=ProjectedGradientDescent,
    eps=adv_train_eps,
    max_iter = 5
)

# Run adv training
# freeze_benign_and_cols keeps rcvTime (col 0) untouched, same threat model as
# adv_test's freeze_cols default, and additionally only lets the attack perturb
# timesteps labeled attacker=1 - a real attacker can't manipulate benign
# messages from other vehicles, so FGSM/PGD shouldn't be allowed to "cheat" by
# moving benign traffic during training either.
print("> Running Adv Training")
attack = freeze_benign_and_cols(
    ProjectedGradientDescent(
        classifier, eps=adv_train_eps, max_iter=5,
        eps_step=2.5 * adv_train_eps / 5
    ),
    freeze_cols=(0,3)
)
trainer = AdversarialTrainer(classifier, attacks=attack, ratio=adv_train_ratio)
trainer.fit(x_train_np, y_train_np, nb_epochs=adv_train_epochs)

# Re-evaluate after adversarial training
print("> After Clean Test")
after_clean_out = clean_data_test(
    model, classifier, x_test, y_test, 
    checkpoint_file=checkpoint_file, data_file=data_file,
    save_path=adv_save_dir,
    filename=f"clean.json",
    save_results=True
)

# print("> After Adv Test")
after_adv_f1 = []
# for i in range(1, 31):
#     eps = float(i/100)
#     after_adv_out = adv_test(
#         classifier, x_test, y_test, 
#         checkpoint_file=checkpoint_file, data_file=data_file,
#         end_index=len(y_test.numpy()),
#         path=adv_save_dir + "/fgsm",
#         filename=f"after_adv_eps_{eps}_advtrained_{name}.json",
#         Attack=FastGradientMethod,
#         eps=eps,
#     )
#     after_adv_f1.append(after_adv_out["metrics"]["f1"])


# Save the adversarially-trained weights (loadable via utils.functions.load_model_checkpoint)
ckpt_out = f"{adv_save_dir}/advtrained_{name}.ckpt"
torch.save(model.learner.state_dict(), ckpt_out)
print(f"Saved adversarially-trained checkpoint to {ckpt_out}")

import json
config = {
    "eps": adv_train_eps,
    "adv_train_epochs": adv_train_epochs,
    "adv_train_ratio": adv_train_ratio,
    "checkpoint_file": checkpoint_file,
    "attack": "ProjectedGradientDescent",
    "name": name,
}

with open(f"{adv_save_dir}/config.json", "w") as f:
    json.dump(config, f, indent=4)
    print("saved:", f"{adv_save_dir}/config.json")


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!
=== Adversarial training: RandPos-PGD-Evasion-30-30-200 | eps:  0.99 ===
> Before Clean Test
torch.Size([124774, 10, 2])
0.9976138469777732
0.959778796870785
Model got 1231979/1247740 right.
Accuracy: 0.9873683619984933, Precision: 0.9976138469777732, Recall: 0.959778796870785, F1 Score: 0.9783306592093668
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
Saved to ../fed/data-test/randpos2_pgd-0.99_epochs-20_trained-30-30-200_ratio-1.0/before_clean_advtrained_RandPos-PGD-Evasion-30-30-200.json
> Before Adv Test
=== Attack: ProjectedGradientDescent, kwargs: {'eps': 0.99, 'max_iter': 5} ===


PGD - Batches:   0%|          | 0/3900 [00:00<?, ?it/s]

Accuracy:           0.1062
Precision:          0.1312
Recall:             0.3574
F1:                 0.1920
ASR (FNR):          0.6426
False Positive Rate:1.0000
TP=132500, TN=7, FP=877033, FN=238200
Time elapsed:       69.82s
Saved metrics to ../fed/data-test/randpos2_pgd-0.99_epochs-20_trained-30-30-200_ratio-1.0/before_adv_eps_0.99_advtrained_RandPos-PGD-Evasion-30-30-200.json
> Running Adv Training


Precompute adv samples:   0%|          | 0/1 [00:00<?, ?it/s]

Adversarial training epochs:   0%|          | 0/20 [00:00<?, ?it/s]

> After Clean Test
torch.Size([124774, 10, 2])
0.9999838008935427
0.8326247639600756
Model got 1185689/1247740 right.
Accuracy: 0.9502692868706621, Precision: 0.9999838008935427, Recall: 0.8326247639600756, F1 Score: 0.9086624303203461
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
Saved to ../fed/data-test/randpos2_pgd-0.99_epochs-20_trained-30-30-200_ratio-1.0/clean.json
Saved adversarially-trained checkpoint to ../fed/data-test/randpos2_pgd-0.99_epochs-20_trained-30-30-200_ratio-1.0/advtrained_RandPos-PGD-Evasion-30-30-200.ckpt
saved: ../fed/data-test/randpos2_pgd-0.99_epochs-20_trained-30-30-200_ratio-1.0/config.json


In [13]:
x_batch = x_train.numpy()[:32]
y_batch = y_train.numpy()[:32]
x_adv = attack.generate(x_batch, y=y_batch)

benign = (y_batch != 1)
assert np.array_equal(x_adv[benign], x_batch[benign]), "benign timesteps were perturbed!"
assert not np.array_equal(x_adv[~benign], x_batch[~benign]), "attacker timesteps weren't touched at all"
assert np.array_equal(x_adv[:, :, (0,3)], x_batch[:, :, (0,3)]), "frozen cols moved"

fully_benign_windows = benign.all(axis=1)
assert np.array_equal(x_adv[fully_benign_windows], x_batch[fully_benign_windows]), "all-benign window changed"
print("max perturbation on attacker rows:", np.abs(x_adv[~benign] - x_batch[~benign]).max(), "(should be <= eps)")


max perturbation on attacker rows: 0.99 (should be <= eps)


In [18]:
diffs = np.abs(x_adv[~benign] - x_batch[~benign])   # only attacker cells, unfrozen cols
print("mean:", diffs.mean(), "median:", np.median(diffs))
print("fraction near eps bound (>0.9):", (diffs > 0.9).mean())


mean: 0.31041077 median: 0.2314905
fraction near eps bound (>0.9): 0.0875


In [9]:
import json
config = {
    "eps": adv_train_eps,
    "adv_train_epochs": adv_train_epochs,
    "adv_train_ratio": adv_train_ratio,
    "checkpoint_file": checkpoint_file,
    "attack": "ProjectedGradientDescent",
    "name": name,
}

with open(f"{adv_save_dir}/config.json", "w") as f:
    json.dump(config, f, indent=4)
    print("saved:", f"{adv_save_dir}/config.json")

saved: ../fed/data-test/randpos_pgd-0.99_epochs-5_trained-30-30-200/config.json


In [ ]:
## Find lowerbound threshold - lowest eps that still
lo_eps = 0.07 # fgsm, from the previously done eps sweep, f1 = 0.7079755805153897 < 0.77


In [ ]:
## Find upperbound threshold - highest eps that starts making benign wrong

import json, time, torch
# When f1 is at least 20 pts down from og (so 0.77)

# ratio = 0.5, epochs = 5, eps = 0.2 -> ran until eps = 0.99, ratio to low? (high-eps-iterative)
# ratio = 1.0, epochs = 5, eps = 0.1 -> clean f1 score dec immediately, ratio to high? (high-eps-iterative-ratio-1.0)
# ratio = 0.7, epochs = 5, eps = 0.1 -> running current (high-eps-iterative-ratio-0.7)

# Defined
clean_f1 = 0.976470669379591
threshold_diff = 0.2
working_eps = 0.9

# Training Hyperparams
adv_train_epochs = 5
adv_train_ratio = 0.7

# Var definitions
x_train_np = x_train.numpy()
y_train_np = y_train.numpy()
name = get_filename_from_path(checkpoint_file)
index = 1

# run once

while index < 2:
    start = time.time()
    print(f"\n\n>>> New eps: {working_eps}")
    # Reload classifier
    index = index + 1
    model, classifier = get_model_classifier(checkpoint_file)
    adv_save_dir = f"../fed/data-test/freeze-col-test/iter-{index}"
    os.makedirs(adv_save_dir, exist_ok=True)
    print(f"Saving to: {adv_save_dir}")

    ## Adv train model
    print("> Running Adv Training")
    attack = freeze_attack_cols(FastGradientMethod(classifier, eps=working_eps), freeze_cols=(0,))
    trainer = AdversarialTrainer(classifier, attack, ratio=adv_train_ratio)
    trainer.fit(x_train_np, y_train_np, nb_epochs=adv_train_epochs, batch_size=64)

    # Save model
    ckpt_out = f"{adv_save_dir}/advtrained_{name}.ckpt"
    torch.save(model.learner.state_dict(), ckpt_out)
    print(f"Saved adversarially-trained checkpoint to {ckpt_out}")

    ## Test model
    after_clean_out = clean_data_test(
        model, classifier, x_test, y_test, 
        checkpoint_file=checkpoint_file, data_file=data_file,
        save_path=adv_save_dir,
        filename=f"after_clean_advtrained_{name}.json",
        save_results=True
    )

    # Save config
    

    # Check if f1 is below
    adv_f1 = after_clean_out["wrapper"]["f1"]
    if (adv_f1 < clean_f1 - threshold_diff): 
        print(f"BREAKING WITH EPS {working_eps}")
        break
    else:
        print(f"f1 = {adv_f1} >  {clean_f1 - threshold_diff}")
        working_eps = round(working_eps + 0.01, 2)
        print(f"Increasing eps to {working_eps}")

print("goodbye world")


    
    

"""
- Start at ~ 0.2 eps
- Adv train model on the # of eps (save the model)
- Run clean test (and SAVE it)
- Check if it's below the threshold

* Can try just doing 10 points.. later (or just see from data)
"""

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.




>>> New eps: 0.9
Checkpoint path exists!
Saving to: ../fed/data-test/freeze-col-test/iter-2
> Running Adv Training


Adversarial training epochs: 100%|██████████| 5/5 [07:05<00:00, 85.16s/it]


Saved adversarially-trained checkpoint to ../fed/data-test/freeze-col-test/iter-2/advtrained_RandomPos-final.ckpt
Saved to ../fed/data-test/freeze-col-test/iter-2/after_clean_advtrained_RandomPos-final.json
f1 = 0.9955288715845922 >  0.7764706693795911
Increasing eps to 0.91
goodbye world


"\n- Start at ~ 0.2 eps\n- Adv train model on the # of eps (save the model)\n- Run clean test (and SAVE it)\n- Check if it's below the threshold\n\n* Can try just doing 10 points.. later (or just see from data)\n"

In [ ]:
# y_train_np.sum(axis=1)[102]

np.int64(0)

In [7]:
## Find upperbound threshold - highest eps that starts making benign wrong

import json, time, torch
import numpy as np
# When f1 is at least 20 pts down from og (so 0.77)

# ratio = 0.5, epochs = 5, eps = 0.2 -> ran until eps = 0.99, ratio to low? (high-eps-iterative)
# ratio = 1.0, epochs = 5, eps = 0.1 -> clean f1 score dec immediately, ratio to high? (high-eps-iterative-ratio-1.0)
# ratio = 0.7, epochs = 5, eps = 0.1 -> running current (high-eps-iterative-ratio-0.7)

# Defined
clean_f1 = 0.976470669379591
threshold_diff = 0.2
working_eps = 0.1

# Training Hyperparams
adv_train_epochs = 5

# Var definitions
x_train_np = x_train.numpy()
y_train_np = (y_train.numpy() == 1).all(axis=1).astype(np.int64)
name = get_filename_from_path(checkpoint_file)
index = 2

# run once

while index < 3:
    start = time.time()
    print(f"\n\n>>> New eps: {working_eps}")
    # Reload classifier
    index = index + 1
    model, classifier = get_model_classifier(checkpoint_file, collapsed=True)
    classifier._reduce_labels = True
    adv_save_dir = f"../fed/data-test/trades-test/iter-{index}"
    os.makedirs(adv_save_dir, exist_ok=True)
    print(f"Saving to: {adv_save_dir}")

    ## Adv train model
    print("> Running Adv Training")
    attack = FastGradientMethod(classifier, eps=working_eps)
    trainer = AdversarialTrainerTRADESPyTorch(classifier, attack=attack, beta=0.6)
    trainer.fit(x_train_np, y_train_np, nb_epochs=adv_train_epochs, batch_size=64)

    # Save model
    ckpt_out = f"{adv_save_dir}/advtrained_{name}.ckpt"
    torch.save(model.learner.state_dict(), ckpt_out)
    print(f"Saved adversarially-trained checkpoint to {ckpt_out}")

    ## Test model
    after_clean_out = clean_data_test(
        model, classifier, x_test, y_test, 
        checkpoint_file=checkpoint_file, data_file=data_file,
        save_path=adv_save_dir,
        filename=f"after_clean_advtrained_{name}.json",
        save_results=True,
        collapsed=True
    )

    # Save config
    config = {
        "eps": working_eps,
        "adv_train_epochs": adv_train_epochs,
        "beta": 0.6,
        "clean_f1_baseline": clean_f1,
        "threshold_diff": threshold_diff,
        "index": index,
        "checkpoint_file": checkpoint_file,
        "name": name,
        "timeElapsedSec": (time.time() - start)
    }

    with open(f"{adv_save_dir}/config.json", "w") as f:
        json.dump(config, f, indent=4)

    # Check if f1 is below
    adv_f1 = after_clean_out["wrapper"]["f1"]
    if (adv_f1 < clean_f1 - threshold_diff): 
        print(f"BREAKING WITH EPS {working_eps}")
        break
    else:
        print(f"f1 = {adv_f1} >  {clean_f1 - threshold_diff}")
        working_eps = round(working_eps + 0.01, 2)
        print(f"Increasing eps to {working_eps}")

print("goodbye world")


    
    

"""
- Start at ~ 0.2 eps
- Adv train model on the # of eps (save the model)
- Run clean test (and SAVE it)
- Check if it's below the threshold

* Can try just doing 10 points.. later (or just see from data)
"""

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.




>>> New eps: 0.1
Checkpoint path exists!
Saving to: ../fed/data-test/trades-test/iter-3
> Running Adv Training


Adversarial Training TRADES - Epochs: 100%|██████████| 5/5 [12:13<00:00, 146.73s/it]


Saved adversarially-trained checkpoint to ../fed/data-test/trades-test/iter-3/advtrained_RandomPos-final.ckpt
Saved to ../fed/data-test/trades-test/iter-3/after_clean_advtrained_RandomPos-final.json
f1 = 0.9970922762743267 >  0.7764706693795911
Increasing eps to 0.11
goodbye world


"\n- Start at ~ 0.2 eps\n- Adv train model on the # of eps (save the model)\n- Run clean test (and SAVE it)\n- Check if it's below the threshold\n\n* Can try just doing 10 points.. later (or just see from data)\n"

In [16]:
min(3, 1)

1